In [ ]:
import requests
import json
import base64
import time
import os
import pandas as pd
from IPython.display import display, Markdown

# ---------- 1. Model feature detection ----------
def get_model_features(model_name):
    """
    Determine if model has thinking/reasoning and vision.
    1. Try to load CSV and look up 'Input' column.
    2. If not found, use name heuristics.
    Returns (has_thinking, has_vision)
    """
    # Try to read combined CSV (if you have it)
    csv_path = r"E:\llm\Model\csv\all_models_combined.csv"
    input_type = None
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            row = df[df['Name'] == model_name]
            if not row.empty:
                input_type = row.iloc[0].get('Input', '')
        except:
            pass

    # Vision detection from CSV if available
    if input_type is not None:
        has_vision = 'Image' in str(input_type)
    else:
        # Heuristic based on name
        low = model_name.lower()
        vision_patterns = ['vl', 'vision', 'llava', 'llama3.2-vision', 'llama4',
                           'gemma3', 'gemma4', 'ministral', 'mistral-small3.1',
                           'mistral-medium-3.5', 'medgemma', 'qwen2.5vl', 'qwen3-vl']
        has_vision = any(p in low for p in vision_patterns)

    # Thinking detection
    low = model_name.lower()
    thinking_patterns = ['thinking', 'reasoning', 'deepseek-r1', 'phi4-reasoning',
                         'phi4-mini-reasoning', 'nemotron-3.5-lightning',
                         'nemotron-3-nano', 'nemotron-3-super']
    has_thinking = any(p in low for p in thinking_patterns)
    # Some models may output reasoning without "thinking" in name (e.g., deepseek-r1)
    if 'deepseek-r1' in low:
        has_thinking = True

    return has_thinking, has_vision

# ---------- 2. Image helpers ----------
def image_to_base64_from_path(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def image_to_base64_from_url(image_url):
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode("utf-8")

def image_to_base64(image_input):
    if image_input.startswith(("http://", "https://")):
        return image_to_base64_from_url(image_input)
    else:
        return image_to_base64_from_path(image_input)

# ---------- 3. Main chat function ----------
def chat_aware(prompt, model="qwen3:4b", image_input=None):
    """
    Chat with automatic handling of thinking/reasoning and vision.
    - If model supports vision and no image_input given, ask user for path/URL.
    - Displays model features in the output.
    """
    has_thinking, has_vision = get_model_features(model)

    # If vision model and no image provided, ask interactively
    # if has_vision and image_input is None:
    #     print(f"🔍 Model '{model}' supports images.")
    #     img = input("Enter image path or URL (or press Enter to skip): ").strip()
    #     if img:
    #         image_input = img

    # Build message
    message = {"role": "user", "content": prompt}
    # if image_input:
    #     try:
    #         image_b64 = image_to_base64(image_input)
    #         message["images"] = [image_b64]
    #     except Exception as e:
    #         print(f"❌ Error loading image: {e}")
    #         return

    payload = {
        "model": model,
        "messages": [message],
        "stream": True,
    }

    resp = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        stream=True,
        timeout=300,
    )
    resp.raise_for_status()

    # Timers
    start_time = time.time()
    first_thinking_time = None
    first_content_time = None
    thinking_text = ""
    content_text = ""

    # Create display handle
    handle = display(Markdown(""), display_id=True)

    # Build feature header
    features = []
    if has_thinking:
        features.append("🧠 **Thinking**")
    if has_vision:
        features.append("👁️ **Vision**")
    features_str = ", ".join(features) if features else "None"
    header = f"### Model: {model}\n**Features:** {features_str}\n\n"

    for line in resp.iter_lines(decode_unicode=True):
        if not line:
            continue
        data = json.loads(line)
        msg = data.get("message", {})

        # Thinking
        if "thinking" in msg and msg["thinking"]:
            thinking_text += msg["thinking"]
            if first_thinking_time is None:
                first_thinking_time = time.time()

        # Content
        if "content" in msg and msg["content"]:
            content_text += msg["content"]
            if first_content_time is None:
                first_content_time = time.time()

        # Build markdown
        md = header
        md +=f"Prompt = {prompt}"
        md += f"\n----\n"
        if thinking_text:
            md += f"## 🤔 Thinking\n\n{thinking_text}\n\n---\n\n"
        md += f"## 💬 Answer\n\n{content_text}"

        handle.update(Markdown(md))

        if data.get("done"):
            break

    total_time = time.time() - start_time

    # Final timings
    timings = ""
    if first_thinking_time:
        timings += f"**TTFT (thinking):** {first_thinking_time - start_time:.3f}s  \n"
    if first_content_time:
        timings += f"**TTFT (content):** {first_content_time - start_time:.3f}s  \n"
    timings += f"**Total time:** {total_time:.3f}s"

    handle.update(Markdown(md + "\n---\n" + timings))

    return thinking_text, content_text

In [ ]:
FILE_NAME = "../NNDesign.pdf"
FILE_NAME_PAPER = "../2608.02980v1.pdf"


In [ ]:
import pymupdf
import os

Folder1 = "border"

os.makedirs(Folder1, exist_ok=True)

doc = pymupdf.open(FILE_NAME_PAPER)

print(f"Total pages: {doc.page_count}")

def choose(pno):
    page = doc[pno]
    blocks = page.get_text("blocks")
    print(f"Page {pno+1}: {len(blocks)} block(s)")
    return blocks
    #for block in blocks:
        # Tuple: (x0, y0, x1, y1, text, block_no, block_type)
        #rect = pymupdf.Rect(block[0], block[1], block[2], block[3])
        #print(block[4])
        #return block
    return blocks
        

In [ ]:
#chat_aware(choose(1)[0][4],model="qwen3:0.6b")

In [ ]:
def choose(pno,model="granite4:350m-h"):
    page = doc[pno]
    blocks = page.get_text("blocks")
    #print(f"Page {pno+1}: {len(blocks)} block(s)")
    #return blocks
    for block in blocks:
        # Tuple: (x0, y0, x1, y1, text, block_no, block_type)
        #rect = pymupdf.Rect(block[0], block[1], block[2], block[3])
        #print(block[4])
        #return block
        chat_aware(block[4],model)
    pass


In [ ]:
#granite4:3b-h
#granite4:1b-h
#granite4:350m-h - don't use for paper and book
#granite4:350m - don't use for paper and book
#granite4.2:3b-q4_K_M
#gemma3:1b
#gemma:2b-instruct-v1.1-q4_K_M   - giving bullet point, but some information is not know to this model.
#gemma2:2b-instruct-q4_K_M 
#gemma4:e2b-it-qat  
#gemma3:270m-it-qat 
#qwen3:0.6b
#qwen3:1.7b
#qwen3:4b
#qwen2:0.5b-instruct                   6f48b936a09f    352 MB   
#qwen2:1.5b                            f6daf2b25194    934 MB   
#qwen2:0.5b                            6f48b936a09f    352 MB   
#qwen:4b                               d53d04290064    2.3 GB   
#qwen:1.8b                             b6e8ec2e7126    1.1 GB   
#qwen:0.5b
#qwen2:1.5b-instruct-q8_0              908c3f054aac    1.6 GB   
#qwen2:1.5b-instruct-q6_K              1fcc9e410c55    1.3 GB   
#qwen2:1.5b-instruct-q5_K_M            2ce7afe8f9e9    1.1 GB   
#qwen2:1.5b-instruct-q5_K_S            272e01e2734a    1.1 GB   
#qwen2:1.5b-instruct-q5_1              5b1266375ca6    1.2 GB   
#qwen2:1.5b-instruct-q5_0              945eea96cfa5    1.1 GB   
#qwen2:1.5b-instruct-q4_K_M            0d2a504f5771    986 MB   
#qwen2:1.5b-instruct-q4_K_S            976868269c51    940 MB   
#qwen2:1.5b-instruct-q4_1              8cbdd0be53c3    1.0 GB   
#qwen2:1.5b-instruct-q4_0              f6daf2b25194    934 MB   
#qwen2:1.5b-instruct-q3_K_L            2c7ac92e9b2a    880 MB   
#qwen2:1.5b-instruct-q3_K_M            4a143b4bca71    824 MB   
#qwen2:1.5b-instruct-q3_K_S            980b3bdef07c    760 MB   
#qwen2:1.5b-instruct-q2_K              7851c5581bce    676 MB   
#qwen2:1.5b-instruct                   f6daf2b25194    934 MB   
#qwen2:0.5b-instruct-fp16              f9b12d5481d2    994 MB   
#qwen2:0.5b-instruct-q8_0              6b8eef84f0bf    531 MB   
#qwen2:0.5b-instruct-q6_K              8e5bba81c7b5    505 MB   
#qwen2:0.5b-instruct-q5_K_M            d9515b0adb44    420 MB   
#qwen2:0.5b-instruct-q5_K_S            dfba5ca4486a    412 MB   
#qwen2:0.5b-instruct-q5_1              76769ded76b2    419 MB   
#qwen2:0.5b-instruct-q5_0              b17625f38956    396 MB   
#qwen2:0.5b-instruct-q4_K_M            19d15f48d098    397 MB   
#qwen2:0.5b-instruct-q4_K_S            f6898a38eeae    385 MB   
#qwen2:0.5b-instruct-q4_1              a1a9a139f5f4    374 MB   
#qwen2:0.5b-instruct-q4_0              6f48b936a09f    352 MB   
#qwen2:0.5b-instruct-q3_K_L            425aee1fa207    369 MB   
#qwen2:0.5b-instruct-q3_K_M            2adbb16f0a5d    355 MB   
#qwen2:0.5b-instruct-q3_K_S            ae41a13fd9b2    338 MB   
#qwen2:0.5b-instruct-q2_K              06dda8c1519a    338 MB   


# Automate Chat Local "gemma3:270m-it-qat" model with Paper 

In [18]:
import pymupdf
import os

Folder1 = "border"

os.makedirs(Folder1, exist_ok=True)

doc = pymupdf.open(FILE_NAME_PAPER)

print(f"Total pages: {doc.page_count}")

for pno in range(doc.page_count):
 choose(pno,model="gemma3:270m-it-qat")

Total pages: 21


### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = Qwen-3D: A Generalist 3D Vision-Language Model for Spatial Understanding

----
## 💬 Answer

The Qwen-3D is a versatile 3D Vision-Language Model (VLM) designed to effectively understand spatial relationships and patterns in 3D scenes. It is built upon a foundation of deep learning, incorporating advanced techniques for semantic understanding, visual reasoning, and contextual understanding. The model aims to provide comprehensive and informative insights into the spatial relationships between objects and scenes.
---
**TTFT (content):** 0.030s  
**Total time:** 10.427s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = Lucy Lin†, Ayush Jain†, Yifan Liu, Katerina Fragkiadaki

----
## 💬 Answer


---
**Total time:** 0.003s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = Carnegie Mellon University

----
## 💬 Answer


---
**Total time:** 0.002s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = {lucylin,ayushj2,yifanliu,kfragki2}@andrew.cmu.edu

----
## 💬 Answer


---
**Total time:** 0.002s

...

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = Text Tokenizer

----
## 💬 Answer

Text Tokenizer

---
**TTFT (content):** 0.001s  
**Total time:** 0.460s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = Visual Tokenizer

----
## 💬 Answer

Visual Tokenizer is a powerful library for converting text to code. It's known for its efficiency, readability, and ability to handle a wide range of text formats, including code.
---
**TTFT (content):** 0.002s  
**Total time:** 4.139s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = (iii)  <REF> Token Decoders

----
## 💬 Answer

The text indicates that there are multiple decoders available for the `REF` token.

---
**TTFT (content):** 0.001s  
**Total time:** 1.979s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

Prompt = Visual Tokenizer

----
## 💬 Answer

Visual Tokenizer is a powerful and

KeyboardInterrupt: 